<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week5/lab1/WPI_week5_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biomedical Segmentation Model Development

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 120-150 minutes.

**Clinical disclaimer:** This lab uses public microscopy nuclei images for segmentation workflow practice. The outputs are not clinical measurements or diagnostic tools.


## Learning Objectives

By the end of this lab, you should be able to:

- Load biomedical image-mask pairs from a public runtime dataset.
- Merge instance masks into a binary semantic segmentation target.
- Compare a classical thresholding baseline with a learned U-Net-style model.
- Train with BCE + Dice loss and save the best validation checkpoint.
- Preserve a model checkpoint so a later Colab session can reuse it.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week5_lab1_responses_LastName_FirstName.docx`

Answer Q1-Q5 in the Word document, using 2-5 sentences per question.


## Workflow

This lab follows a biomedical segmentation development pipeline:

`Microscopy image -> Nuclei masks -> Classical baseline -> U-Net -> Validation Dice/IoU -> Portable checkpoint`

The dataset is downloaded at runtime. No raw images, masks, or trained checkpoints are committed to GitHub.


## Setup

Run this setup cell first. It installs small Python dependencies, clones the public course helper repo in Colab, and applies WPI plot styling.


In [ ]:
#@title Setup course environment
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scikit-image",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    repo_dir = Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])
else:
    subprocess.check_call(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"])
    subprocess.check_call(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"])

sys.path.insert(0, str(repo_dir))
import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "wpi_ai_bootcamp" or module_name.startswith("wpi_ai_bootcamp."):
        sys.modules.pop(module_name)

import random
import shutil
import numpy as np
import matplotlib.pyplot as plt

from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, disk, remove_small_objects

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

from wpi_ai_bootcamp.data import load_bbbc038_nuclei_segmentation_subset
from wpi_ai_bootcamp.notebook import make_wpi_overlay, setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)


## Data Loading

This notebook uses **BBBC038 / 2018 Data Science Bowl nuclei segmentation** from the Broad Bioimage Benchmark Collection. Each image has many per-nucleus mask PNG files. The course loader merges those instance masks into one binary foreground mask for this lab.


In [ ]:
images, masks, metadata = load_bbbc038_nuclei_segmentation_subset(
    max_samples=160,
    image_size=128,
    download=True,
    random_state=42,
)

source = metadata["source"]
print(source.name)
print(source.url)
print(source.license_note)
print("images:", images.shape, images.dtype, float(images.min()), float(images.max()))
print("masks:", masks.shape, masks.dtype, sorted(np.unique(masks).tolist()))


## Hyperparameters

Only change values in this block when the notebook asks you to run a controlled comparison. The defaults are chosen to give a meaningful validation comparison in Colab while keeping runtime classroom-friendly.


In [ ]:
# STUDENT-EDITABLE HYPERPARAMETERS
SEED = 42
LR = 8e-4
BATCH_SIZE = 12
EPOCHS = 8
BASE_CHANNELS = 16
THRESHOLD = 0.5
MORPH_RADIUS = 2
MIN_OBJECT = 20
SHOW_EXAMPLE_INDEX = 0

# TODO: For Part 5, change exactly one value above and record the result in your Word response.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


## Part 1 - Dataset Visualization

Inspect one microscopy image and its merged nuclei mask before training anything.


In [ ]:
def show_image_mask_overlay(image, mask, title="Sample"):
    gray = image.squeeze()
    binary = mask.squeeze() > 0.5
    overlay = make_wpi_overlay(gray, binary)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title("Microscopy image")
    axes[1].imshow(binary, cmap="gray")
    axes[1].set_title("Merged nuclei mask")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_mask_overlay(images[SHOW_EXAMPLE_INDEX], masks[SHOW_EXAMPLE_INDEX], "BBBC038 example")


### Part 1 Assessment - Dataset Inspection (20 pts)

Required notebook output: one image, merged mask, and overlay figure.

Word response Q1: Why does merging per-nucleus instance masks create a binary semantic segmentation target?

Grading criteria: correct output, clear mask interpretation, and a concise segmentation explanation.


## Part 2 - Classical Baseline Segmentation

Use Otsu thresholding plus morphology as a reproducible baseline. This baseline can work well on some microscopy images, so the learned model must earn its comparison.


In [ ]:
class NucleiArrayDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        if self.augment:
            if torch.rand(()) > 0.5:
                image = torch.flip(image, dims=[2])
                mask = torch.flip(mask, dims=[2])
            if torch.rand(()) > 0.5:
                image = torch.flip(image, dims=[1])
                mask = torch.flip(mask, dims=[1])
            brightness = 0.9 + 0.2 * torch.rand(())
            image = torch.clamp(image * brightness, 0.0, 1.0)
        return image, mask


def classical_segmentation(image_tensor):
    image_np = image_tensor.squeeze().cpu().numpy()
    threshold = threshold_otsu(image_np)
    pred = image_np > threshold
    pred = binary_closing(pred, disk(MORPH_RADIUS))
    pred = remove_small_objects(pred, min_size=MIN_OBJECT)
    return pred.astype(np.float32)

full_dataset = NucleiArrayDataset(images, masks, augment=False)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_base, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
train_dataset = NucleiArrayDataset(images[train_base.indices], masks[train_base.indices], augment=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
print("Train samples:", len(train_dataset), "Validation samples:", len(val_dataset))


In [ ]:
baseline_dice = []
baseline_iou = []
for image, mask in val_dataset:
    pred = classical_segmentation(image)
    gt = mask.squeeze().numpy()
    baseline_dice.append(dice_score_np(gt, pred))
    baseline_iou.append(iou_score_np(gt, pred))

print("Validation baseline Dice:", float(np.mean(baseline_dice)))
print("Validation baseline IoU :", float(np.mean(baseline_iou)))


### Part 2 Assessment - Classical Baseline (20 pts)

Required notebook output: validation Dice and IoU for the classical baseline.

Word response Q2: Why can a thresholding baseline be strong on some microscopy images and weak on others?

Grading criteria: metrics are produced, the baseline is described accurately, and one limitation is identified.


## Part 3 - U-Net Training With Quality Tracking

Train a compact U-Net-style model using BCE + Dice loss. The notebook saves the best validation Dice checkpoint, not just the final epoch.


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(),
        )
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        up = self.up(e2)
        return self.out(self.dec1(torch.cat([up, e1], dim=1)))


def dice_score_np(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    inter = (gt * pred).sum()
    return float((2 * inter + eps) / (gt.sum() + pred.sum() + eps))


def iou_score_np(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    inter = (gt * pred).sum()
    union = ((gt + pred) > 0).sum()
    return float((inter + eps) / (union + eps))


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = (probs * targets).sum(dim=dims)
    denom = probs.sum(dim=dims) + targets.sum(dim=dims)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def segmentation_loss(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits, targets) + dice_loss(logits, targets)


In [ ]:
model = TinyUNet(BASE_CHANNELS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
train_losses = []
val_dice_history = []
best_val_dice = -1.0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch_images, batch_masks in train_loader:
        batch_images = batch_images.to(DEVICE)
        batch_masks = batch_masks.to(DEVICE)
        logits = model(batch_images)
        loss = segmentation_loss(logits, batch_masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()

    avg_loss = running / len(train_loader)
    train_losses.append(avg_loss)

    model.eval()
    epoch_dice = []
    with torch.no_grad():
        for val_image, val_mask in val_loader:
            logits = model(val_image.to(DEVICE))
            pred = (torch.sigmoid(logits).cpu().numpy().squeeze() > THRESHOLD).astype(np.float32)
            gt = val_mask.numpy().squeeze()
            epoch_dice.append(dice_score_np(gt, pred))
    mean_val_dice = float(np.mean(epoch_dice))
    val_dice_history.append(mean_val_dice)
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {avg_loss:.4f} - val Dice: {mean_val_dice:.4f}")

model.load_state_dict(best_state)
print("Best validation Dice:", round(best_val_dice, 4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(1, EPOCHS + 1), train_losses, marker="o", color=WPI_COLORS["crimson"])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training loss")
axes[0].set_title("BCE + Dice Loss")
axes[0].grid(True, alpha=0.3)
axes[1].plot(range(1, EPOCHS + 1), val_dice_history, marker="o", color=WPI_COLORS["accent_green"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Dice")
axes[1].set_ylim(0, 1)
axes[1].set_title("Validation Quality")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 3 Assessment - U-Net Training (20 pts)

Required notebook output: training loss curve, validation Dice curve, and best validation Dice.

Word response Q3: Why is saving the best validation checkpoint better than saving only the final epoch?

Grading criteria: training completes, validation quality is tracked, and the response connects checkpointing to generalization.


## Part 4 - Evaluation And Checkpoint Save

Evaluate the best model, compare it to the classical baseline, and save a checkpoint that Lab 2 can reuse.


In [ ]:
def predict_mask(model, image_tensor, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor.unsqueeze(0).to(DEVICE))
        prob = torch.sigmoid(logits).cpu().squeeze().numpy()
    return (prob > threshold).astype(np.float32), prob

unet_dice = []
unet_iou = []
saved_example = None
for i, (image, mask) in enumerate(val_dataset):
    pred, prob = predict_mask(model, image, THRESHOLD)
    gt = mask.squeeze().numpy()
    unet_dice.append(dice_score_np(gt, pred))
    unet_iou.append(iou_score_np(gt, pred))
    if i == min(SHOW_EXAMPLE_INDEX, len(val_dataset) - 1):
        saved_example = (image, mask, classical_segmentation(image), pred, prob)

mean_baseline_dice = float(np.mean(baseline_dice))
mean_baseline_iou = float(np.mean(baseline_iou))
mean_unet_dice = float(np.mean(unet_dice))
mean_unet_iou = float(np.mean(unet_iou))
print("Validation baseline Dice:", mean_baseline_dice)
print("Validation baseline IoU :", mean_baseline_iou)
print("Validation U-Net Dice   :", mean_unet_dice)
print("Validation U-Net IoU    :", mean_unet_iou)

if mean_unet_dice <= mean_baseline_dice:
    print("Quality note: the U-Net did not beat the baseline in this run. For your submitted run, try EPOCHS=12 or BASE_CHANNELS=24 and report the comparison.")
else:
    print("Quality note: the U-Net beat the classical baseline on validation Dice in this run.")


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Baseline", "U-Net"], [mean_baseline_dice, mean_unet_dice], color=[WPI_COLORS["gray"], WPI_COLORS["crimson"]])
plt.ylabel("Dice score")
plt.ylim(0, 1)
plt.title("Validation Dice Comparison")
plt.tight_layout()
plt.show()

image, mask, baseline_pred, unet_pred, prob = saved_example
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(image.squeeze(), cmap="gray")
axes[0].set_title("Image")
axes[1].imshow(mask.squeeze(), cmap="gray")
axes[1].set_title("Target")
axes[2].imshow(baseline_pred, cmap="gray")
axes[2].set_title("Baseline")
axes[3].imshow(prob, cmap="magma")
axes[3].set_title("U-Net probability")
axes[4].imshow(unet_pred, cmap="gray")
axes[4].set_title("U-Net mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_DIR = Path("week5_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
checkpoint_path = OUTPUT_DIR / "best_unet.pt"
checkpoint = {
    "model_state_dict": model.state_dict(),
    "base_channels": BASE_CHANNELS,
    "threshold": THRESHOLD,
    "image_size": int(images.shape[-1]),
    "data_source": source.name,
    "loader": "load_bbbc038_nuclei_segmentation_subset",
    "max_samples": int(images.shape[0]),
    "random_state": 42,
    "validation_metrics": {
        "baseline_dice": mean_baseline_dice,
        "baseline_iou": mean_baseline_iou,
        "unet_dice": mean_unet_dice,
        "unet_iou": mean_unet_iou,
    },
}
torch.save(checkpoint, checkpoint_path)
print("Saved local checkpoint:", checkpoint_path)

DRIVE_CHECKPOINT = Path("/content/drive/MyDrive/WPI_AI_Bootcamp/week5/best_unet.pt")
if Path("/content/drive/MyDrive").exists():
    DRIVE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_path, DRIVE_CHECKPOINT)
    print("Copied checkpoint to Google Drive:", DRIVE_CHECKPOINT)
else:
    print("Google Drive is not mounted. Run the optional Drive/download cell before ending this Colab session.")


### Optional Checkpoint Persistence Cell

Run this cell if you want to download the checkpoint or mount Google Drive. Lab 2 can use either the downloaded file or the Drive copy in a fresh Colab session.


In [ ]:
# Optional: download and/or save the Lab 1 checkpoint for a future Colab session.
try:
    from google.colab import files
    files.download(str(checkpoint_path))
    print("Download started for:", checkpoint_path)
except Exception as exc:
    print("Download helper unavailable outside Colab:", type(exc).__name__)

# Optional: mount Drive, then copy the checkpoint to a persistent folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_path, DRIVE_CHECKPOINT)
    print("Saved persistent Drive checkpoint:", DRIVE_CHECKPOINT)
except Exception as exc:
    print("Drive save skipped:", type(exc).__name__, str(exc)[:120])


### Part 4 Assessment - Evaluation And Portable Checkpoint (20 pts)

Required notebook output: metric comparison, qualitative prediction figure, and saved `week5_outputs/best_unet.pt`.

Word response Q4: Did the learned model outperform the baseline in your run? Use Dice and the overlay to support your answer.

Grading criteria: metrics and plots are present, checkpoint is saved, and the response treats quality as evidence rather than assuming training worked.


## Part 5 - One Controlled Comparison

Change exactly one parameter and compare the validation metric. Keep all other settings identical.

Recommended first comparison: rerun the notebook with only `EPOCHS` changed to 12, then compare best validation Dice.


In [ ]:
# TODO: Record one controlled comparison in your Word response.
controlled_comparison = {
    "parameter_changed": "EPOCHS",
    "original_value": EPOCHS,
    "new_value": "12 in a rerun",
    "baseline_dice_this_run": mean_baseline_dice,
    "unet_dice_this_run": mean_unet_dice,
}
controlled_comparison


### Part 5 Assessment - Controlled Comparison (20 pts)

Required notebook output: original and comparison values for Dice and/or IoU.

Word response Q5: Which single parameter did you change, and how did it affect validation quality and runtime?

Grading criteria: exactly one parameter changes, metrics are compared fairly, and the interpretation is tied to the observed result.


## Optional Challenge

Try changing `BASE_CHANNELS` to 24 and rerun from the start. Compare runtime, best validation Dice, and visual mask quality.


## Attribution

- Data: BBBC038 / 2018 Data Science Bowl nuclei segmentation dataset, downloaded at runtime from the Broad Bioimage Benchmark Collection.
- Dataset page: https://bbbc.broadinstitute.org/BBBC038
- Download used: `stage1_train.zip` from the BBBC038 page.
- Recommended citation: Caicedo, J. C., Goodman, A., Karhohs, K. W. et al. *Nucleus segmentation across imaging experiments: the 2018 Data Science Bowl*. Nature Methods 16, 1247-1253 (2019).
- License note: The BBBC038 page lists the image set copyright as CC0.
- Libraries: NumPy, Matplotlib, scikit-image, PyTorch, and course helper code.
